## Human in the loop design pattern

#### Import libraries

In [ ]:
import os, asyncio
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel
from openai import AsyncOpenAI


load_dotenv(override=True)

#### Define LLM model

In [ ]:
#client = AsyncOpenAI(base_url="http://localhost:11434/v1")
#model = OpenAIChatCompletionsModel(model="gpt-oss", openai_client=client)
model = "gpt-4.1-nano"

#### Define Tools for Agents

In [ ]:
# Tool for agent to annotate variants (mocked)
@function_tool
def annotate_variant(variant: str) -> str:
    """Annotate a genetic variant for clinical significance (mocked)."""
    return f"Variant {variant}: Likely pathogenic (mocked annotation)."

#### Define Annotation Agent

In [ ]:
# Agent to draft annotations
annotation_agent = Agent(
    name="VariantAnnotationAgent",
    instructions="Given a list of genetic variants, use your tool to annotate each for clinical significance.",
    model=model,
    tools=[annotate_variant],
)

#### Define Human-in-the-loop function workflow

In [ ]:
# Human-in-the-loop function (simulated for demo)
async def human_review(report: str) -> bool:
    print("Human review required. Draft report:")
    print(report)
    # Simulate human input (replace with actual UI or input in real use)
    feedback = input("Approve report? (y/n): ")
    return feedback.lower().startswith("y")

# HITL workflow
async def hitl_pipeline(variants):
    variants_str = ", ".join(variants)
    draft = await Runner.run(annotation_agent, variants_str)
    approved = await human_review(draft.final_output)
    if approved:
        print("Report finalized and submitted.")
        return draft.final_output
    else:
        print("Report rejected. Please revise and resubmit.")
        # In real use, agent could revise based on human feedback
        return None



#### Execute workflow

In [ ]:
from agents import trace
with trace("hitl_pipeline_demo"):    # Example usage
    variants = ["BRCA1:c.68_69del", "TP53:c.215C>G"]
    final_report = await hitl_pipeline(variants)
    print("Final Report:", final_report)